# Install requirements and dependencies

In [ ]:
PROJECT_PATH = "/home/jupyter-gabriel/projects/pricing_automation/pricing-automation/etl_pipeline"

In [ ]:
%pip install -r "$PROJECT_PATH/requirements.txt"

## Load kedro with magic command

This will create 3 objects available in the enviroment:
1. session (a kedro.framework.session object which is capable of creating new sessions, running pipelines or nodes)
2. context (a kedro.framework.context object which contains, among other specification, the resolve catalog)
3. catalog (an object capable of loading and saving the catalog entries defined in the catalog.yml)


In [ ]:
%load_ext kedro.ipython

Change the notebook to the project path and reload the project from there

In [ ]:
%cd $PROJECT_PATH
%reload_kedro .

## (Optional) Load custom functions
If you want to manually test the functions you build load them

You can manually run each step and take advantage of the Kedro Catalog Utility to load and save datasets specified in the catalog

OR, you can get rid of Kedro entirely by making manual loadings and savings knowing that each function requires its own inputs and outputs

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pricing_automation.pipelines.utils import *
from pricing_automation.pipelines.a01_aoi_period import *
from pricing_automation.pipelines.n01_extract_data import *
from pricing_automation.pipelines.n02_process_data import *
from pricing_automation.pipelines.n03_create_triggers import *
from pricing_automation.pipelines.n04_bootstrap_aep import *
from pricing_automation.pipelines.n05_pricing_quote import *
from pricing_automation.pipelines.n06_visualizations import *

# Create a Session

This lets you set quick parameters to overrride the ones in the yml files

### Step 0: Define parameters to override at runtime

In [ ]:
%reload_kedro
runtime_params = {
    'country': 'bolivia', # Local folder name to store data 
    'region': 'valles', # new parameter to work with a segment of the country with many leads within this area
    'lead_id': 'cidre', # Lead name
    'loc_id': 'ID-081',
    'provider': 'ERA5', # Data provider, can be 'ERA5' or 'UCSB'
    'field': 'swc', # Short variable name, can be 'swc', 'prcp', 'tmin', 'tmax'
    'start_year': 2002, # (optional) Use this to override the initial year of data
    'end_year': 2025, # (optional) Use this to override the ending year of data
}
session_trial = session.create(
    runtime_params = runtime_params
)
# (Optional) Load the catalog with the context provided earlier for quick loads
catalog_trial = session_trial.load_context().catalog 

Now you can call and load any entry in the Catalog, provided it exists, by running:

__catalog_trial.load('namespace.entry')__

### Step 1. Get Area Of Interest

It either creates a bounding box area with the parameters in __params.yml__ (provided the _override_ parameter is set to __true__

Otherwise it will attempt to create a bounding box from the _gdf_request_ entry in the __catalog.yml__)

You can use a large geometry and subset it on the fly. To do this use the _include_ and _exclude_ parameter in _params_s_ in the  __params.yml__ file. You should provide a dictionary of the form 

* 'column': ['value1', 'value2']

E.g.: For Argentina the Province of Buenos Aires can be filtered with 'depto' = '06' so use:

* 'include': {'depto': ['06']}

The same logic applies if it is easier to exclude some values. E.g.: Excluding the Province of Buenos Aires

* 'exclude': {'depto': ['06']}

In [ ]:
%reload_kedro 
# reload_kedro is optional, but use it if you modify any part of the project.
# This way, it reloads the project before running anything, e.g. changing a parameter.
session.create(
    runtime_params=runtime_params
).run(
    pipeline_names = ['planet'],
    node_names = ['get_aoi']
)

The previous cell run a single node with its name, these can be found in: 

___src/pricing_automation/pipelines/pipeline.py___

There, you will also see tags under each entry, this groups some nodes that can or need to be run in sequence. We can specify a list of tags to run:

E.g.
session.run(tags=['extract'])

Available tags are the following:

1. 'extract': runs nodes _get_aoi_ and _extract_data_
2. 'process': runs nodes _process_data_request_ and _summarize_processed_data_
3. 'triggers': runs nodes _generate_triggers_
4. 'aep': runs nodes _run_bootstrap_aep_
5. 'pricing' runs nodes _run_bootstrap_aep_, _run_pricing_quote_ and _plot_aep_

You can run either of these nodes independently

E.g.
session.run(node_names=['namespace.get_aoi'])

___Namespaces___ are a way of running diffent pipelines or branches of pipelines under the same project.

For example, this project registered 3 namespaces: 'swc', 'prcp', 'temp' 

The only difference between them is in the _process_data_request_ node that performs different operations in each of them.

(in the __pipeline_registry.py__ file)

### Step 2. Extract data

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    pipeline_names = ['planet'],
    node_names = ['get_aoi', 'extract_data']
)

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    pipeline_names = ['planet'],
    node_names = ['extract_data_planet']
)

### Step 3. Create climate areas

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    pipeline_names = ['planet'],
    node_names = ['create_climate_areas']
)

### Step 4. Preprocess data

In [ ]:
gdf_aoi = catalog_trial.load('gdf_aoi')

In [ ]:
list_mun = ['Tarija', 'Quillacollo', 'Mizque', 'Colomi']

In [ ]:
list_ids = gdf_aoi[gdf_aoi['municipio'].isin(list_mun)]['location_id'].unique()

In [ ]:
%reload_kedro
for id in list_ids:
    print(f'Location ID: {id}')
    runtime_params['loc_id'] = id
    session.create(
        runtime_params = runtime_params
    ).run(
        pipeline_names = ['planet'],
        #tags=['process']
        node_names = ['preprocess_data_1']
    )

In [ ]:
%reload_kedro
for id in list_ids:
    print(f'Location ID: {id}')
    runtime_params['loc_id'] = id
    session.create(
        runtime_params = runtime_params
    ).run(
        pipeline_names = ['planet'],
        node_names = ['preprocess_data_2']
    )

### Step 5. Process data

In [ ]:
%reload_kedro
for id in list_ids[1:]:
    print(f'Location ID: {id}')
    runtime_params['loc_id'] = id
    session.create(
        runtime_params = runtime_params
    ).run(
        pipeline_names = ['planet'],
        node_names = ['process_data']
    )

### Step 6. Create triggers

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    pipeline_names = ['planet'],
    tags=['triggers']
)

### Step 7. Create activation map

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    pipeline_names = ['planet'],
    node_names = ['viz']
)

### Step 8. Run bootstrap and pricing quote

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    pipeline_names = ['planet'],
    tags=['pricing']
)

# Complete Run

There is no need to run each step part by part, Kedro handles dependencies at runtime.

So you can specify a list of tags to run or just run the whole pipeline from step 1 to step 7.

The _extract_data_ takes the longest to complete (about an hour for 30 years of data if AOI is ~ 4 by 4 degrees in size).

The rest of the steps are relatively faster (under 3 minutes each and some take seconds)

In [ ]:
%reload_kedro
session.create(
    runtime_params = runtime_params
).run(
    pipeline_names = ['planet'],
    #tags=['extract', 'process', 'triggers', 'pricing']
)